In [1]:
import pandas as pd

# Read in dataset from data preprocessing
df = pd.read_csv("clean_matches.csv")
df

,match_id,date,ground,home team,home score,away team,away score,goalscorer(s)
0,1,2006-07-26,Murrayfield,Hearts,3,Siroki Brijeg,0,"['Branimir Arnic (og)', 'Ibrahim Tall', 'Roman..."
1,2,2008-01-01,Old Trafford,Man United,1,Birmingham,0,['Carlos Tevez']
2,3,2008-11-01,Old Trafford,Man United,4,Hull,3,"['Cristiano Ronaldo (x2)', 'Nemanja Vidic', 'M..."
3,4,2008-11-19,Windsor Park,NI,0,Hungary,2,"['Sandor Torghelle', 'Zoltan Gera']"
4,5,2010-10-30,Camp Nou,Barcelona,5,Sevilla,0,"['Lionel Messi (x2)', 'David Villa (x2)', 'Dan..."
5,6,2012-02-29,Windsor Park,NI,0,Norway,3,"['Harvard Nordtveit', 'Tarik Elyounoussi', 'Es..."
6,7,2012-08-04,Portman Road,Ipswich,3,West Ham,1,"['Michael Chopra', 'Luke Chambers', 'Jason Sco..."
7,8,2012-08-08,Old Trafford,Brazil,3,South Korea,0,"['Romulo', 'Leandro Damiao (x2)']"
8,9,2016-01-19,Villa Park,Aston Villa,2,Wycombe,0,"['Cieran Clark', 'Idrissa Gana Gueye']"
9,10,2016-03-28,Windsor Park,NI,1,Slovenia,0,['Conor Washington']


In [2]:
# Function that builds the search query that will be input to Google
import calendar

def build_search_query(match):                              # Match will be row from the dataset
    # Set up variables that will be used in the string for search query
    home_team = match["home team"]
    away_team = match["away team"]
    month = calendar.month_name[int(match["date"][5:7])]    # Calendar converts the numbers 1-12 to the corresponding months January-December
    year = match["date"][0:4]
    return f"{home_team} vs {away_team} in {month} {year}"  # Return the search query in an f-string

In [3]:
# Test function from previous cell
build_search_query(df.iloc[44])

'Cambridge vs Salford in March 2026'

In [4]:
# Add the seach query as a column in the database
df["search query"] = df.apply(build_search_query, axis=1)
df

,match_id,date,ground,home team,home score,away team,away score,goalscorer(s),search query
0,1,2006-07-26,Murrayfield,Hearts,3,Siroki Brijeg,0,"['Branimir Arnic (og)', 'Ibrahim Tall', 'Roman...",Hearts vs Siroki Brijeg in July 2006
1,2,2008-01-01,Old Trafford,Man United,1,Birmingham,0,['Carlos Tevez'],Man United vs Birmingham in January 2008
2,3,2008-11-01,Old Trafford,Man United,4,Hull,3,"['Cristiano Ronaldo (x2)', 'Nemanja Vidic', 'M...",Man United vs Hull in November 2008
3,4,2008-11-19,Windsor Park,NI,0,Hungary,2,"['Sandor Torghelle', 'Zoltan Gera']",NI vs Hungary in November 2008
4,5,2010-10-30,Camp Nou,Barcelona,5,Sevilla,0,"['Lionel Messi (x2)', 'David Villa (x2)', 'Dan...",Barcelona vs Sevilla in October 2010
5,6,2012-02-29,Windsor Park,NI,0,Norway,3,"['Harvard Nordtveit', 'Tarik Elyounoussi', 'Es...",NI vs Norway in February 2012
6,7,2012-08-04,Portman Road,Ipswich,3,West Ham,1,"['Michael Chopra', 'Luke Chambers', 'Jason Sco...",Ipswich vs West Ham in August 2012
7,8,2012-08-08,Old Trafford,Brazil,3,South Korea,0,"['Romulo', 'Leandro Damiao (x2)']",Brazil vs South Korea in August 2012
8,9,2016-01-19,Villa Park,Aston Villa,2,Wycombe,0,"['Cieran Clark', 'Idrissa Gana Gueye']",Aston Villa vs Wycombe in January 2016
9,10,2016-03-28,Windsor Park,NI,1,Slovenia,0,['Conor Washington'],NI vs Slovenia in March 2016


In [ ]:
# API key needs to be kept private from public GITHUB repository
# Key stored in an .env file
# .env file listed within .gitignore
# This cell reads in the key and stores it with api_key

from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("SERPAPI_KEY")

In [ ]:
# Function that performs the Google search query
# Uses SerpAPI

from serpapi import GoogleSearch

def get_urls(query):
    # Parameters for the Google search
    params = {
        "q": query,                     # The string to be query                
        "api_key": api_key,             # The SerpAPI key
        "num": 20                       # Max 20 links, but will naturally restrict to the first page of Google searches
    }

    search = GoogleSearch(params)       
    results = search.get_dict()         # Sends request to the API

    return [r["link"] for r in results.get("organic_results", [])]          # Extract links

# Test the function get_urls on the query generated for one of the matches
# Need to be careful about overusing this function: get 250 queries per month through free tier of SerpAPI

urls = get_urls("Man United vs Birmingham in January 2008")
urls

In [ ]:
# Access the urls variable from the previous cell without needing to send another query to SERPAPI
urls

In [5]:
# Able to redefine urls from above cell if I start a new session without needing to re-use a SERPAPI search
urls = ['https://www.youtube.com/watch?v=givC4ZvbqYc',
 'https://www.skysports.com/football/manchester-united-vs-birmingham-city/teams/94538',
 'https://www.transfermarkt.us/manchester-united_birmingham-city/index/spielbericht/81687',
 'https://www.manutd.com/en/videos/detail/man-utd-1-birmingham-city-0-extended-highlights-premier-league-2007-08',
 'https://www.statmuse.com/fc/ask/full-time-scores-of-man-u-vs-birmingham-in-premiere-league-match-in-2008',
 'http://news.bbc.co.uk/sport2/hi/football/eng_prem/7163892.stm',
 'https://www.premierleague.com/en/match/128642/manchester-united-vs-birmingham-city',
 'https://www.espn.ph/football/report/_/gameId/220076',
 'https://www.footballcritic.com/premier-league-manchester-united-fc-birmingham-city-fc/match-stats/42194']

In [6]:
# Test on one match before expanding to all to preserve SerpAPI queries
# Remove this cell before roll-out

df = df.iloc[[1]].copy()        # Make a copy of the second entry in a sample dataframe
df["article urls"] = [urls]     # Add a new column of the list of url strings
df

,match_id,date,ground,home team,home score,away team,away score,goalscorer(s),search query,article urls
1,2,2008-01-01,Old Trafford,Man United,1,Birmingham,0,['Carlos Tevez'],Man United vs Birmingham in January 2008,"[https://www.youtube.com/watch?v=givC4ZvbqYc, ..."


In [7]:
# Read text from an articles

import requests
from bs4 import BeautifulSoup
import re

def extract_article(url):
    html = requests.get(url).text               # Gets the raw HTML code of the webpage
    soup = BeautifulSoup(html, "html.parser")   # Parses the HTML code into paragraphs/heading/etc. by looking for <p>/<h1>/etc.

    paragraphs = [p.get_text() for p in soup.find_all("p")]     # Gets all paragraph text
    text = "\n".join(paragraphs)
    text = re.sub(r"\s+", " ", text)             # Find all chunks of whitespace and replace them with a single space. \s is any whitespace character. \s+ means more than one in a row
    return text.strip()

# Testing the function
extract_article("http://news.bbc.co.uk/sport2/hi/football/eng_prem/7163892.stm")

'Tevez\'s overhead kick and a Cristiano Ronaldo back-heel carved open the Blues defence and the Argentine calmly slotted home the winner on 25 minutes. Cameron Jerome was down injured then but Birmingham complaints were limited. Tevez, who went off late on with an ankle injury, also hit the woodwork twice while the Blues never gave up. Interview: Manchester United\'s Rio Ferdinand Interview: Birmingham boss Alex McLeish United manager Sir Alex Ferguson, sat in the stands to serve out his two-match touchline ban, made five changes to his starting line-up. Ronaldo partnered Tevez in the centre of attack with Wayne Rooney still absent with a virus and Ryan Giggs not even on the bench. Birmingham boss Alex McLeish - once a protege of Ferguson in their days together at Aberdeen - started with two up front. 606: DEBATE Have your say on Birmingham\'s narrow defeat at Old Trafford Jerome, partnering Gary O\'Connor, heeded the positive mantra of his boss and scorched a long-range effort over To

In [8]:
# Determine the source of an article from the url found by the Google search

from urllib.parse import urlparse

# A set of all trusted sources: aim is that found urls will include one of these within domain name
# NEEDS UPDATING TO INCLUDE MORE SOURCES
trusted_sources = {
    "bbc",
    "skysports",
    "espn",
    "rte"
}

def get_source(url):
    domain = urlparse(url).netloc.lower()               # netloc extracts the network location: i.e. "https://www.example.com:8080/page" goes to "www.example.com:8080"
    for name in trusted_sources:
        if name in domain:                              # Check if one of the trusted sources is in the domain name
            return name
    return "other"

# Testing the function
get_source("http://news.bbc.co.uk/sport2/hi/football/eng_prem/7163892.stm")

'bbc'

In [9]:
# Build article records into a Dataframe

# Create a list with an entry for each article from a trusted source for each match
article_records = []
for _, row in df.iterrows():                        # iterrows() iterates through tuples (*,*) of row index and corresponding one row Dataframe
    match_id = row["match_id"]
    urls = row["article urls"]
    for url in urls:
        source = get_source(url)
        text = extract_article(url)[:32700]         # Limit the number of characters in the text: Excel can only store up to approx 32700 characters before going to a new cell
        
        # Add source, url and text for each trusted article for a given match
        if source in trusted_sources and text:      
            article_records.append({
                "match_id":match_id,
                "source":source,
                "url": url,
                "text": text
            })

# Convert data to a dataframe and store into a csv file
article_df = pd.DataFrame(article_records)
article_df.to_csv("articles.csv", index=False)